# Network-condition robustness - DoH vs classic DNS under degraded links

**Purpose:** this is the project's primary novelty. Almost all published DoH latency studies measure a clean link; here every `profile` (clean, moderate, high_jitter, lossy) was produced with `tc netem`, and we measure how each method degrades.

Outputs: `results/netem_summary.csv`, `results/fig_netem_rtt.png`.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# find the repo root (the folder that contains client/) no matter where Jupyter was started
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "client").is_dir())
for sub in ("client", "ml", "capture"):
    sys.path.insert(0, str(ROOT / sub))
RESULTS = ROOT / "results"
print("repo root:", ROOT)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from summarize import load_raw, summarize, summarize_by_run
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

df = load_raw(RESULTS)
PREFERRED = ["clean", "moderate", "high_jitter", "lossy"]
profiles = [p for p in PREFERRED if p in set(df.profile)] + sorted(set(df.profile) - set(PREFERRED))
print("profiles found:", profiles)
print(df.groupby("profile").run_id.nunique().rename("runs"))

## 1. Summary table (all queries) - saved for the paper

In [ ]:
s = summarize(df)
s.to_csv(RESULTS / "netem_summary.csv", index=False)
s.round(2)

## 2. Mean ± std across repeated runs
Queries inside one run are correlated, so the error bars use the spread **between runs**.

In [ ]:
byrun = summarize_by_run(df)
resolvers = sorted(df.resolver.unique())
fig, axes = plt.subplots(1, len(resolvers), figsize=(5.2 * len(resolvers), 4), sharey=True, squeeze=False)
for ax, res in zip(axes[0], resolvers):
    for method, g in byrun[byrun.resolver == res].groupby("method"):
        g = g.set_index("profile").reindex(profiles).dropna(subset=["mean_of_runs_ms"])
        ax.errorbar(g.index, g.mean_of_runs_ms, yerr=g.std_of_runs_ms.fillna(0), marker="o", capsize=3, label=method)
    ax.set_title(res); ax.set_xlabel("network profile"); ax.grid(alpha=.3)
axes[0][0].set_ylabel("RTT (ms), mean ± std across runs"); axes[0][-1].legend(title="method")
fig.tight_layout(); fig.savefig(RESULTS / "fig_netem_rtt.png", dpi=300); plt.show()

## 3. The cost of encryption, in milliseconds
`doh_cold_overhead` = what the first lookup over a new DoH connection costs vs plain UDP; `doh_warm_overhead` = steady-state cost on a reused connection.

In [ ]:
piv = s.pivot_table(index=["profile", "resolver"], columns="method", values="mean_ms")
piv["doh_cold_overhead_ms"] = piv["doh_cold"] - piv["classic_udp"]
piv["doh_warm_overhead_ms"] = piv["doh_warm"] - piv["classic_udp"]
piv = piv.reset_index(); piv["profile"] = pd.Categorical(piv.profile, profiles, ordered=True)
piv.sort_values(["resolver", "profile"]).round(2)

## 4. The handshake: cold vs warm DoH
`handshake_overhead_ms` = doh_cold − doh_warm = what setting up TCP + TLS adds to ONE lookup. `doh_warm_vs_udp_ms` = the steady-state price of encryption once the connection exists.

In [ ]:
from summarize import handshake_overhead
ho = handshake_overhead(df); ho["profile"] = pd.Categorical(ho.profile, profiles, ordered=True)
ho.sort_values(["resolver", "profile"]).round(2)

## 5. Jitter
Mean absolute difference between successive RTTs of the same method (higher = more erratic). Netem's `high_jitter` profile should make this jump.

In [ ]:
from summarize import jitter
jt = jitter(df); jt["profile"] = pd.Categorical(jt.profile, profiles, ordered=True)
jt.sort_values(["resolver", "method", "profile"]).round(2)

## 6. Answer consistency between methods
Do classic DNS and DoH return the same IPs? Big sites rotate IPs between queries, so identical_rate below 1.0 is normal - what matters is `rcode_agree_rate` and `overlap_rate`, and whether the **degraded** profiles differ from `clean`.

In [ ]:
from summarize import consistency
cs = consistency(df); cs["profile"] = pd.Categorical(cs.profile, profiles, ordered=True)
cs.sort_values(["resolver", "profile"]).round(3)

## 7. Reliability: success rate per method and profile
UDP has no retransmission, TCP/TLS does - on lossy links this is where they differ most.

In [ ]:
sr = s.pivot_table(index=["profile", "resolver"], columns="method", values="success_rate").reset_index()
sr["profile"] = pd.Categorical(sr.profile, profiles, ordered=True)
sr.sort_values(["resolver", "profile"]).round(3)